# Evaluate existing H11 checkpoints locally

This notebook never trains. It evaluates the existing KL-off checkpoints in `h11_probe/runs` on TweetEval, Banking77, and STS-B.


In [3]:
from pathlib import Path
import json
import os
import subprocess
import sys

repo_root = Path.cwd()
if not (repo_root / "experiment.py").exists():
    raise FileNotFoundError("Open this notebook from the repository root")
os.chdir(repo_root)
print("Repository:", repo_root)


Repository: /Users/manoz/Acads/sem 7/NLP/A3


In [4]:
import pandas as pd

def required_path(relative):
    path = repo_root / relative
    if not path.exists():
        raise FileNotFoundError(path)
    return path

corpus = required_path("data/final_data.csv")
tweet_train = required_path("data/tweet_train.csv")
tweet_test = required_path("data/tweet_test.csv")
banking_train = required_path("data/banking/banking_train.csv")
banking_test = required_path("data/banking/banking77_test.csv")
stsb_test = repo_root / "data/stsb_test.csv"
if not stsb_test.exists():
    import csv
    import io
    import urllib.request
    import zipfile
    with urllib.request.urlopen("https://dl.fbaipublicfiles.com/glue/data/STS-B.zip", timeout=60) as response:
        archive = zipfile.ZipFile(io.BytesIO(response.read()))
    with archive.open("STS-B/dev.tsv") as handle:
        parsed = list(csv.reader(io.StringIO(handle.read().decode("utf-8")), delimiter="\t"))
    rows = [
        {"sentence1": fields[7], "sentence2": "\t".join(fields[8:-1]), "score": fields[-1]}
        for fields in parsed[1:] if len(fields) >= 10
    ]
    stsb_test = repo_root / "stsb_validation_local.csv"
    pd.DataFrame(rows)[["sentence1", "sentence2", "score"]].to_csv(stsb_test, index=False)

kl_off_dir = required_path("layerwise_probe/runs")
seeds = [1, 2, 42]
models = ["mrl", "mrl_sia", "mrl_pic", "mipic"]
for seed in seeds:
    for model in models:
        checkpoint = kl_off_dir / f"seed{seed}" / "checkpoints" / model / "best.pt"
        if not checkpoint.exists():
            raise FileNotFoundError(checkpoint)
print("Using H11 checkpoints:", kl_off_dir)
print("Seeds:", seeds)


Using H11 checkpoints: /Users/manoz/Acads/sem 7/NLP/A3/h11_probe/runs
Seeds: [1, 2, 42]


In [5]:
import shutil

if shutil.which("pytest"):
    subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_experiment.py"], check=True)
else:
    print("pytest not installed; skipping tests")


..                                                                       [100%]
2 passed in 10.48s


In [6]:
OUTPUT_DIR = repo_root / "h11_probe_local_evaluation"
command = [
    sys.executable, "experiment.py",
    "--corpus", str(corpus),
    "--tweet-train", str(tweet_train),
    "--tweet-test", str(tweet_test),
    "--banking-train", str(banking_train),
    "--banking-test", str(banking_test),
    "--stsb-test", str(stsb_test),
    "--output-dir", str(OUTPUT_DIR),
    "--seeds", *[str(seed) for seed in seeds],
    "--kl-off-only",
    "--kl-off-dir", str(kl_off_dir),
    "--skip-training",
]
print("Running evaluation only:", " ".join(command))
subprocess.run(command, check=True)


Running evaluation only: /Users/manoz/Acads/sem 7/NLP/A3/.venv/bin/python experiment.py --corpus /Users/manoz/Acads/sem 7/NLP/A3/data/final_data.csv --tweet-train /Users/manoz/Acads/sem 7/NLP/A3/data/tweet_train.csv --tweet-test /Users/manoz/Acads/sem 7/NLP/A3/data/tweet_test.csv --banking-train /Users/manoz/Acads/sem 7/NLP/A3/data/banking/banking_train.csv --banking-test /Users/manoz/Acads/sem 7/NLP/A3/data/banking/banking77_test.csv --stsb-test /Users/manoz/Acads/sem 7/NLP/A3/stsb_validation_local.csv --output-dir /Users/manoz/Acads/sem 7/NLP/A3/h11_probe_local_evaluation --seeds 1 2 42 --kl-off-only --kl-off-dir /Users/manoz/Acads/sem 7/NLP/A3/h11_probe/runs --skip-training


W1107 01:42:23.362000 61872 .venv/lib/python3.15/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 52339.87it/s]
[transformers] BertModel LOAD REPORT from: huawei-noah/TinyBERT_General_6L_768D
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4, 5, 6}.bias      | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4, 5, 6}.weight    | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTE

 PICModule initialized with 5 pipeline pairs:
   (layer_1, dim_16) ? (layer_2, dim_32)
   (layer_2, dim_32) ? (layer_3, dim_64)
   (layer_3, dim_64) ? (layer_4, dim_256)
   (layer_4, dim_256) ? (layer_5, dim_512)
   (layer_5, dim_512) ? (layer_6, dim_768)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 49378.59it/s]
[transformers] BertModel LOAD REPORT from: huawei-noah/TinyBERT_General_6L_768D
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4, 5, 6}.bias      | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4, 5, 6}.weight    | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect i

 PICModule initialized with 5 pipeline pairs:
   (layer_1, dim_16) ? (layer_2, dim_32)
   (layer_2, dim_32) ? (layer_3, dim_64)
   (layer_3, dim_64) ? (layer_4, dim_256)
   (layer_4, dim_256) ? (layer_5, dim_512)
   (layer_5, dim_512) ? (layer_6, dim_768)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 49395.53it/s]
[transformers] BertModel LOAD REPORT from: huawei-noah/TinyBERT_General_6L_768D
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4, 5, 6}.bias      | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4, 5, 6}.weight    | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect i

... (4 outputs left out)

 PICModule initialized with 5 pipeline pairs:
   (layer_1, dim_16) ? (layer_2, dim_32)
   (layer_2, dim_32) ? (layer_3, dim_64)
   (layer_3, dim_64) ? (layer_4, dim_256)
   (layer_4, dim_256) ? (layer_5, dim_512)
   (layer_5, dim_512) ? (layer_6, dim_768)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 46881.53it/s]
[transformers] BertModel LOAD REPORT from: huawei-noah/TinyBERT_General_6L_768D
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4, 5, 6}.bias      | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4, 5, 6}.weight    | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect i

 PICModule initialized with 5 pipeline pairs:
   (layer_1, dim_16) ? (layer_2, dim_32)
   (layer_2, dim_32) ? (layer_3, dim_64)
   (layer_3, dim_64) ? (layer_4, dim_256)
   (layer_4, dim_256) ? (layer_5, dim_512)
   (layer_5, dim_512) ? (layer_6, dim_768)
Saved scores and contrasts under /Users/manoz/Acads/sem 7/NLP/A3/h11_probe_local_evaluation


CompletedProcess(args=['/Users/manoz/Acads/sem 7/NLP/A3/.venv/bin/python', 'experiment.py', '--corpus', '/Users/manoz/Acads/sem 7/NLP/A3/data/final_data.csv', '--tweet-train', '/Users/manoz/Acads/sem 7/NLP/A3/data/tweet_train.csv', '--tweet-test', '/Users/manoz/Acads/sem 7/NLP/A3/data/tweet_test.csv', '--banking-train', '/Users/manoz/Acads/sem 7/NLP/A3/data/banking/banking_train.csv', '--banking-test', '/Users/manoz/Acads/sem 7/NLP/A3/data/banking/banking77_test.csv', '--stsb-test', '/Users/manoz/Acads/sem 7/NLP/A3/stsb_validation_local.csv', '--output-dir', '/Users/manoz/Acads/sem 7/NLP/A3/h11_probe_local_evaluation', '--seeds', '1', '2', '42', '--kl-off-only', '--kl-off-dir', '/Users/manoz/Acads/sem 7/NLP/A3/h11_probe/runs', '--skip-training'], returncode=0)

In [7]:
scores = OUTPUT_DIR / "scores.json"
contrasts = OUTPUT_DIR / "contrasts.csv"
print(scores)
print(contrasts)
display(pd.DataFrame(json.loads(scores.read_text())))
display(pd.read_csv(contrasts))


/Users/manoz/Acads/sem 7/NLP/A3/h11_probe_local_evaluation/scores.json
/Users/manoz/Acads/sem 7/NLP/A3/h11_probe_local_evaluation/contrasts.csv


,seed,kl,model,dataset,dimension,score
0,1,off,mrl,tweet,16,62.590375
1,1,off,mrl,tweet,32,65.672969
2,1,off,mrl,tweet,64,67.391869
3,1,off,mrl,tweet,128,68.581552
4,1,off,mrl,tweet,256,70.151532
...,...,...,...,...,...,...
247,42,off,mipic,stsb,64,70.461425
248,42,off,mipic,stsb,128,71.794729
249,42,off,mipic,stsb,256,72.866174
250,42,off,mipic,stsb,512,73.308876


,dataset,kl,dimension,quantity,seed_values,mean,ci_low,ci_high
0,banking77,off,16,interaction,"[6.2131974342864655, -9.84976607710842, 15.129...",3.831111,-27.615729,35.277951
1,banking77,off,16,sia_effect,"[0.20334248421099943, -15.309486480479904, 15....",-0.004089,-37.769797,37.761619
2,banking77,off,32,interaction,"[8.110553064214585, -8.75320268045062, 21.2788...",6.878738,-30.517170,44.274645
3,banking77,off,32,sia_effect,"[2.073666458647292, -12.927014112167168, 18.11...",2.419297,-36.139522,40.978116
4,banking77,off,64,interaction,"[10.241075990639171, -5.241449452204854, 20.68...",8.561768,-23.843524,40.967060
5,banking77,off,64,sia_effect,"[2.483376929465024, -10.536025705793307, 15.05...",2.333424,-29.451446,34.118293
6,banking77,off,128,interaction,"[10.463902815331217, -1.397833995380914, 19.10...",9.389881,-16.178894,34.958656
7,banking77,off,128,sia_effect,"[2.7291714187730065, -5.74737649412949, 12.413...",3.131622,-19.441515,25.704759
8,banking77,off,256,interaction,"[7.246634155814178, 0.6308553355921731, 14.877...",7.584860,-10.124898,25.294617
9,banking77,off,256,sia_effect,"[0.4277400190634921, -3.5048927051772196, 9.95...",2.293009,-14.901376,19.487393
